# Notebook 09 — auditoria de features (F1–F6 + SHAP)

Objetivo: testar as 6 pistas de features do `docs/plano_acima05.md` §2 com o mesmo protocolo do NB08 (mesmas folds seed 42, Macro-F1, XGB 500/6/0.05). Nada aqui entra no pipeline sem delta 5/5 folds. NB02 só se consolida depois.

| Track | Ideia | Estado |
|---|---|---|
| T0 | base XGB 339 (referência própria) | — |
| T1 | `N_SEM_RESPOSTA` de `train_nb01.csv` (fecha §7.3) | — |
| T2 | 3 contagens N/A por bloco (trabalho/repro/outras) | — |
| T3 | tabaco: status/intensidade/início (SK2/IN45/SK3/5/6) | — |
| T4 | `O_ANSWERED` + `IDADE_MAXIMA`=55 (gate oficial Q-O) | — |
| T5 | ablação por blocos (MH/PN/SS/cuidados/DH/PE/AL) | — |
| T6+SHAP | auditoria IN15 + top-gain + SHAP global/por-classe | — |

Modo rápido: `NB09_MODO=rapido` (2 folds, smoke test). A decisão é sempre no modo completo.

In [ ]:
import os, json, time, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
K = 16
TARGET = 'AGE_RANGE'

def _data_dir():
    for cand in [os.path.join('..', '..', 'data'), os.path.join('..', 'data'), 'data']:
        if os.path.isdir(cand):
            return cand
    raise FileNotFoundError('pasta data/ nao encontrada')
DATA_DIR = _data_dir()

MODO = os.environ.get('NB09_MODO', 'completo')
SUF = '_rap' if MODO in ('rapido', 'ecra') else ''
N_SPLITS = 2 if SUF else 5

train = pd.read_csv(os.path.join(DATA_DIR, 'train_nb02.csv'))
y = train[TARGET].to_numpy().astype(int)
XBASE = train.drop(columns=['PID', TARGET])
print('MODO=', MODO, '| X:', XBASE.shape, '| PID/HHID em X:', [c for c in ('PID', 'HHID') if c in XBASE.columns])
FOLDS = list(StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE).split(XBASE, y))

def Xmat(df):
    return df.to_numpy(dtype=np.float64)

def xgb_recipe():
    return XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
        tree_method='hist', random_state=RANDOM_STATE, n_jobs=-1, eval_metric='mlogloss')

def train_oof(Xm, tag):
    p = os.path.join(DATA_DIR, f'nb09_{tag}{SUF}.npz')
    if os.path.exists(p):
        d = np.load(p)
        print(f'{tag}: cache {p}')
        return d['f1s'], d['oof'], d['proba']
    t0 = time.time()
    f1s, oof, proba = [], np.zeros(len(y), dtype=int), np.zeros((len(y), K))
    for fi, (tr, va) in enumerate(FOLDS):
        m = xgb_recipe()
        m.fit(Xm[tr], y[tr], sample_weight=compute_sample_weight('balanced', y[tr]))
        pr = m.predict(Xm[va])
        oof[va] = pr
        proba[va] = m.predict_proba(Xm[va])
        f1s.append(f1_score(y[va], pr, average='macro'))
        print(f'  fold {fi}: {f1s[-1]:.4f} ({time.time()-t0:.0f}s)', flush=True)
    f1s = np.array(f1s)
    np.savez_compressed(p, f1s=f1s, oof=oof, proba=proba)
    return f1s, oof, proba

def gravar(nome, obj):
    json.dump(obj, open(os.path.join(DATA_DIR, f'nb09_{nome}{SUF}.json'), 'w'), indent=1, default=float)

def carregar(nome):
    p = os.path.join(DATA_DIR, f'nb09_{nome}{SUF}.json')
    return json.load(open(p)) if os.path.exists(p) else None

RESULTADOS = []
def registar(exp, variante, f1s, base_f1s, nota=''):
    f1s = np.asarray(f1s, float); base_f1s = np.asarray(base_f1s, float)
    r = {'exp': exp, 'variante': variante, 'f1': float(f1s.mean()), 'std': float(f1s.std()),
         'delta': float(f1s.mean() - base_f1s.mean()), 'folds_ganhos': int((f1s > base_f1s).sum()), 'nota': nota}
    RESULTADOS.append(r)
    print(r)
    return r

def tabela(rows):
    df = pd.DataFrame(rows).sort_values('f1', ascending=False)
    for c in ('f1', 'std', 'delta'):
        df[c] = df[c].round(4)
    return df

## T0 — base XGB própria (referência de deltas)

In [ ]:
f1s_base, oof_base, proba_base = train_oof(Xmat(XBASE), 'base')
print(f'BASE: {f1s_base.mean():.4f} +/- {f1s_base.std():.4f}')
try:
    fr = np.load(os.path.join(DATA_DIR, 'xgb_oof.npz'))['oof'].astype(int)
    print('congelado xgb_oof.npz (5-fold):', round(f1_score(y, fr, average='macro'), 4))
except Exception as e:
    print('sem referencia congelada:', e)

## T1 — `N_SEM_RESPOSTA` (fecha §7.3)

Conta `{'Sem resposta','NR'}` por linha em `train_nb01.csv` (espelho de `N_NAO_APLICAVEL`). Guarda também o teste para futura consolidação no NB02.

In [ ]:
nb01 = pd.read_csv(os.path.join(DATA_DIR, 'train_nb01.csv'))
te01 = pd.read_csv(os.path.join(DATA_DIR, 'test_nb01.csv'))
CODS = ['Sem resposta', 'NR']
idc = [c for c in ('PID', 'HHID', TARGET) if c in nb01.columns]
n_sr = nb01.drop(columns=idc).isin(CODS).sum(axis=1).to_numpy()
idc_te = [c for c in ('PID', 'HHID') if c in te01.columns]
n_sr_te = te01.drop(columns=idc_te).isin(CODS).sum(axis=1).to_numpy()
assert len(n_sr) == len(XBASE) and len(n_sr_te) == 3655
print('media:', round(n_sr.mean(), 3), '| >20:', int((n_sr > 20).sum()), '| max:', int(n_sr.max()))
np.savez_compressed(os.path.join(DATA_DIR, f'nb09_t1_counts{SUF}.npz'), train=n_sr, test=n_sr_te)
X1 = pd.concat([XBASE, pd.Series(n_sr, index=XBASE.index, name='N_SEM_RESPOSTA')], axis=1)
f1s, _, _ = train_oof(Xmat(X1), 't1_semresposta')
registar('T1', '+ N_SEM_RESPOSTA', f1s, f1s_base)

## T2 — contagens N/A por bloco

Comprime os sinais de routing N/A em 3 contagens (trabalho / reprodutivo / resto) em vez de dummies esparsas.

In [ ]:
NA_WORK = ['JOBSTAT_COD_Não aplicável']
NA_REPRO = ['IN21_Não aplicável', 'IN22_Não aplicável']
NA_OTHER = ['IN5_Não aplicável', 'IN14_Não aplicável', 'IN16_Não aplicável', 'IN17_Não aplicável',
            'IN44_Não aplicável', 'IN49_Não aplicável', 'IN62_Não aplicável', 'IC2_Não aplicável']
for c in NA_WORK + NA_REPRO + NA_OTHER:
    assert c in XBASE.columns, c
X2 = XBASE.copy()
X2['N_NA_WORK'] = XBASE[NA_WORK].sum(axis=1)
X2['N_NA_REPRO'] = XBASE[NA_REPRO].sum(axis=1)
X2['N_NA_OTHER'] = XBASE[NA_OTHER].sum(axis=1)
f1s, _, _ = train_oof(Xmat(X2), 't2_nabloco')
registar('T2', '+ 3 contagens N/A por bloco', f1s, f1s_base)

## T3 — tabaco: auditoria + construtos

`SK2` (intensidade, sentinela −1), `IN45` (início/anos, sentinela −1), estados em `SK2_ESTADO`/`IN45_ESTADO` (−1 não perguntado, 0 respondeu, −2 recusou). Primeiro a auditoria por classe, depois 5 colunas.

In [ ]:
for c in ['SK2', 'IN45', 'SK3', 'SK5', 'SK6']:
    v = XBASE[c].to_numpy()
    m = v != -1
    s = pd.Series(v[m])
    g = pd.Series(y[m])
    print(f'{c}: n_resp={int(m.sum())} media_resp={s.mean():.2f}')
    print(s.groupby(g).mean().round(2).to_string())
    print()
sk2 = XBASE['SK2'].to_numpy()
in45 = XBASE['IN45'].to_numpy()
X3 = XBASE.copy()
X3['SMOKER_STATUS'] = np.where(XBASE['SK2_ESTADO'].to_numpy() == 0, 2,
    np.where(XBASE['IN45_ESTADO'].to_numpy() == 0, 1, 0))
X3['SK2_VAL'] = np.where(sk2 == -1, 0, sk2)
X3['SK2_ANS'] = (sk2 != -1).astype(int)
X3['IN45_VAL'] = np.where(in45 == -1, 0, in45)
X3['IN45_ANS'] = (in45 != -1).astype(int)
print(pd.crosstab(y, X3['SMOKER_STATUS'], normalize='index').round(2).to_string())
f1s, _, _ = train_oof(Xmat(X3), 't3_tabaco')
registar('T3', '+ 5 construtos tabaco', f1s, f1s_base)

## T4 — `O_ANSWERED` + `IDADE_MAXIMA` (gate oficial Q-O)

Questionário: idade ≤55 → O1/O2, >55 → P1.1. Responder o bloco reprodutivo ⇒ classe ≤11. Como feature (nunca veto — máscaras custam −0.01).

In [ ]:
est_cols = [c for c in ['IN24_ESTADO', 'IN25_ESTADO', 'IN26_ESTADO'] if c in XBASE.columns]
assert est_cols, 'sem colunas ESTADO do bloco O'
o_ans = np.zeros(len(y), dtype=bool)
for c in est_cols:
    o_ans |= (XBASE[c].to_numpy() == 0)
print('O_ANSWERED por classe (frac):')
print(pd.crosstab(pd.Series(y), pd.Series(o_ans), normalize='index').round(3).to_string())
X4 = XBASE.copy()
X4['O_ANSWERED'] = o_ans.astype(int)
X4['IDADE_MAXIMA'] = np.where(o_ans, 55, 99)
f1s, _, _ = train_oof(Xmat(X4), 't4_idademax')
registar('T4', '+ O_ANSWERED + IDADE_MAXIMA', f1s, f1s_base)

## T5 — ablação por blocos

Delta negativo = o bloco faz falta. Sem reter nada: só localiza peso morto e sinal escondido.

In [ ]:
BLOCOS = {'MENTAL': ['MH'], 'PAIN': ['PN'], 'SUPPORT': ['SS'],
          'CARE_USE': ['AM', 'HO', 'MD', 'IC', 'PA1', 'NT'],
          'FOOD': ['DH'], 'EXERCISE': ['PE'], 'ALCOHOL': ['AL']}
for nome, prefs in BLOCOS.items():
    drop = [c for c in XBASE.columns if any(c.startswith(p) for p in prefs)]
    print(f'{nome}: drop {len(drop)} cols')
    f1s, _, _ = train_oof(Xmat(XBASE.drop(columns=drop)), f't5_no_{nome.lower()}')
    registar('T5', f'sem bloco {nome} ({len(drop)} cols)', f1s, f1s_base)

## T6 + SHAP — auditoria IN15, top-gain e atribuições

`IN15` vale 4.56% do ganho LGBM e ninguém sabe porquê. SHAP (TreeExplainer, CPU) global top-25 + top-10 nas piores classes {1, 4, 13}.

In [ ]:
full = xgb_recipe()
full.fit(Xmat(XBASE), y, sample_weight=compute_sample_weight('balanced', y))
imp = pd.Series(full.feature_importances_, index=XBASE.columns).sort_values(ascending=False)
print(imp.head(15).round(4).to_string())
gravar('t6_topgain', imp.head(30).round(5).to_dict())
print()
print('IN15 por classe (frac por linha):')
print(pd.crosstab(pd.Series(y), pd.Series(XBASE['IN15'].to_numpy()), normalize='index').round(3).to_string())
import shap
SHAP_N = 150 if SUF else 1000
rng = np.random.RandomState(RANDOM_STATE)
bg_idx = rng.choice(len(y), size=min(200, len(y)), replace=False)
sm_idx = rng.choice(len(y), size=min(SHAP_N, len(y)), replace=False)
t0 = time.time()
ex = shap.TreeExplainer(full)
sv = ex.shap_values(Xmat(XBASE)[sm_idx])
sv = np.stack(sv, axis=-1) if isinstance(sv, list) else np.asarray(sv)
print('shap_values:', sv.shape, f'({time.time()-t0:.0f}s)')
meanabs = np.abs(sv).mean(axis=0)
glob = meanabs.mean(axis=1)
top25 = pd.Series(glob, index=XBASE.columns).sort_values(ascending=False).head(25)
print(top25.round(5).to_string())
gravar('shap_top25', top25.round(6).to_dict())
for cl in [1, 4, 13]:
    pc = pd.Series(meanabs[:, cl], index=XBASE.columns).sort_values(ascending=False).head(10)
    print(f'--- top SHAP classe {cl}:')
    print(pc.round(5).to_string())
in15_i = list(XBASE.columns).index('IN15')
gravar('shap_in15', {'meanabs_por_classe': meanabs[in15_i].round(6).tolist()})

## Resultados - vereditos (modo completo, 5 folds seed 42)

Base propria: 0.4725. T7a menos 28 cols: 0.4791, +0.0066, 5/5 - ADOTADO no NB02 sec 4.1.T7b com adds: 0.4745, 2/5 - rejeitado. T4 +0.0019 (4/5) e T1 +0.0028 (3/5): arquivados.FOOD fica. Tabaco neutro mas tabelas vao para o relatorio. SHAP: IN15 no topo (0.44).

## T7 — drops combinados (confirmacao dos vencedores T5)

T7a: menos EXERCISE+PAIN+SUPPORT+MENTAL (28 cols). T7b: T7a + vencedores T1/T2/T4. Só se adota com 5/5 ou ganho claro 4/5.

In [ ]:
DROP28 = [c for c in XBASE.columns if any(c.startswith(p) for p in ['PE','PN','SS','MH'])]
print('drop28 n=', len(DROP28))
X7a = XBASE.drop(columns=DROP28)
f1s, _, _ = train_oof(Xmat(X7a), 't7_drop28')
registar('T7', '-28 cols (EXERCISE+PAIN+SUPPORT+MENTAL)', f1s, f1s_base)
X7b = X7a.copy()
X7b['N_SEM_RESPOSTA'] = X1['N_SEM_RESPOSTA']
X7b['N_NA_WORK'] = X2['N_NA_WORK']; X7b['N_NA_REPRO'] = X2['N_NA_REPRO']; X7b['N_NA_OTHER'] = X2['N_NA_OTHER']
X7b['O_ANSWERED'] = X4['O_ANSWERED']; X7b['IDADE_MAXIMA'] = X4['IDADE_MAXIMA']
f1s, _, _ = train_oof(Xmat(X7b), 't7_full')
registar('T7', '-28 cols + T1/T2/T4 (6 feats)', f1s, f1s_base)


In [ ]:
final = tabela(RESULTADOS)
print(final.to_string(index=False))
final.to_csv(os.path.join(DATA_DIR, f'nb09_resultados{SUF}.csv'), index=False)